In [35]:
import pandas as pd
import re
from dataclasses import dataclass
from typing import Dict, List, Any

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
#讀取我的csv
df = pd.read_csv("complaints_final.csv")
df.head()

In [37]:
#確認欄位與筆數
print(df.columns.tolist())
print("資料筆數：", len(df))

['Complaint ID', 'Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Consumer complaint narrative in Chinese', 'Solution in Chinese', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?']
資料筆數： 86


In [ ]:
#篩選我需要的資料
required_cols = [
    "Complaint ID",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Consumer complaint narrative in Chinese",
    "Solution in Chinese"
]

df = df[required_cols].copy()
df = df.fillna("")
df = df.drop_duplicates(subset=["Consumer complaint narrative in Chinese"]).reset_index(drop=True)
df.head()

In [ ]:
#文字清理

#使用set()
stopwords = {
    "的", "了", "呢", "嗎", "我", "想", "請問", "一下", "如何", "怎麼", "要", "該", "是不是",
    "a", "an", "the", "is", "are", "to", "for", "and", "or", "in", "on", "with"
}

#輸入參數 text，預期是字串 str，回傳值也是str
def normalize_text(text: str) -> str:
    text = str(text).lower().strip() #強制轉成字串、全部小寫、移除前後空白
    
    #不包含英文字母、數字、底線 & \u4e00-\u9fff 中文漢字Unicode範圍
    #意思就是 把標點符號、特殊符號都換成空格
    # re.sub(規則, 替換內容, 原字串)
    text = re.sub(r"[^\w\u4e00-\u9fff\s]", " ", text)
    
    #把連續多個空白壓成一個空白
    text = re.sub(r"\s+", " ", text)
    return text

#把文字切成一個個 token（詞）
def tokenize(text: str) -> List[str]:
    text = normalize_text(text)
    
    #抓一段連續的中文漢字，至少 1 個字
    zh_blocks = re.findall(r"[\u4e00-\u9fff]+", text)
    #抓一段連續的英文字母、數字或底線，至少 1 個字元
    en_tokens = re.findall(r"[a-zA-Z0-9_]+", text)

    zh_tokens = []
    for block in zh_blocks:
        block = block.strip()
        if len(block) == 1:
            zh_tokens.append(block)
        else:
            for n in [2, 3, 4]:
                for i in range(len(block) - n + 1):
                    zh_tokens.append(block[i:i+n])

    all_tokens = zh_tokens + en_tokens
    return [t for t in all_tokens if t not in stopwords and len(t) > 1]

In [ ]:
#把solution去重，可能某些金額、數字不同，被視為是不同的方案
def normalize_solution_text(text: str) -> str:
    text = str(text).lower().strip()

    # 把金額統一改成 "金額"
    text = re.sub(r"\$[\d,]+(\.\d+)?", "金額", text)

    # 把數字統一改成 "數字"
    text = re.sub(r"\d+(\.\d+)?", "數字", text)

    # 把不要的符號換成空白
    text = re.sub(r"[^\w\u4e00-\u9fff\s]", " ", text)

    # 空白壓縮
    text = re.sub(r"\s+", " ", text).strip()

    return text

def solution_similarity(sol1: str, sol2: str) -> float:
    tokens1 = set(tokenize(normalize_solution_text(sol1)))
    tokens2 = set(tokenize(normalize_solution_text(sol2)))

    if not tokens1 or not tokens2:
        return 0.0

    return len(tokens1 & tokens2) / max(len(tokens1), 1)

In [41]:
# 關鍵詞重疊分數 > 問題裡的重要關鍵詞，有多少也出現在案例文件裡。

def keyword_overlap_score(question: str, case_doc: str) -> float:
    q_tokens = set(tokenize(question)) #把使用者的問題先斷詞(之前已定義)，再轉成 set
    c_tokens = set(tokenize(case_doc))
    
    if not q_tokens or not c_tokens:
        return 0.0
    
    #採用交集，找出「問題」和「案例」共同擁有的詞
    return len(q_tokens & c_tokens) / len(q_tokens)

In [40]:
#把每一筆客訴資料的幾個欄位合併成一段完整文字，再存進新欄位 case_document
 
def build_case_document(row: pd.Series) -> str:
    #參數row是dataframe的一筆資料，由pd.Series組成，最後函式會回傳str
    parts = [
        str(row.get("Issue", "")),
        str(row.get("Sub-issue", "")),
        str(row.get("Consumer complaint narrative in Chinese", "")),
        #把中文欄位重複一次，這樣中文敘述和中文解法會更容易被抓到。
        str(row.get("Consumer complaint narrative in Chinese", "")),
        str(row.get("Solution in Chinese", "")),
        str(row.get("Solution in Chinese", ""))
    ]
    return "\n".join([p for p in parts if p.strip() != ""])

df["case_document"] = df.apply(build_case_document, axis=1) #對df套用某個函式

df[[
    "Complaint ID",
    "Issue",
    "Sub-issue",
    "case_document"
]].head()

,Complaint ID,Issue,Sub-issue,case_document
0,17987894,Managing an account,Deposits and withdrawals,Managing an account\nDeposits and withdrawals\...
1,17939613,Managing an account,Deposits and withdrawals,Managing an account\nDeposits and withdrawals\...
2,17877488,Managing an account,Deposits and withdrawals,Managing an account\nDeposits and withdrawals\...
3,17998026,Managing an account,Deposits and withdrawals,Managing an account\nDeposits and withdrawals\...
4,18049411,Managing an account,Deposits and withdrawals,Managing an account\nDeposits and withdrawals\...


In [42]:
#建立一個「案例排名結果專用的小容器」

@dataclass #裝飾器（decorator）下面 class 主要是拿來裝資料的，請幫我自動產生一些常用功能。
class RankedCase:
    index: int
    complaint_id: Any #不限型別
    issue: str
    sub_issue: str
    similarity_score: float
    keyword_score: float
    final_score: float

In [44]:
def find_best_category(question: str, df: pd.DataFrame):
    category_texts = (
        df[["Issue", "Sub-issue"]]
        .fillna("")
        .astype(str)
        .drop_duplicates()
        .reset_index(drop=True)
    )

    category_texts["category_text"] = category_texts["Issue"] + " | " + category_texts["Sub-issue"]

    corpus = category_texts["category_text"].tolist() + [question]

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
    matrix = vectorizer.fit_transform(corpus)

    category_vectors = matrix[:-1]
    question_vector = matrix[-1]

    similarities = cosine_similarity(question_vector, category_vectors).flatten()
    best_idx = similarities.argmax()

    best_issue = category_texts.loc[best_idx, "Issue"]
    best_sub_issue = category_texts.loc[best_idx, "Sub-issue"]

    return best_issue, best_sub_issue

In [ ]:
#核心檢索函式
#先自定義 > 相似度 0.7 ; 關鍵詞 0.3

def rank_cases(question: str, df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    
    #如果沒有找到case_document，主動中止程式，並回報一個錯誤訊息
    if "case_document" not in df.columns:
        raise ValueError("df 裡沒有 case_document 欄位，請先執行 build_case_document 那一格。")
    
    best_issue, best_sub_issue = find_best_category(question, df)

    filtered_df = df[
        (df["Issue"] == best_issue) &
        (df["Sub-issue"] == best_sub_issue)
    ].copy()

    #如果那個分類的資料太少，不夠挑出 top_k 筆，就改成從全部案例裡找
    if len(filtered_df) < top_k:
        filtered_df = df.copy()

    docs = df["case_document"].tolist() #把case文字變成一個list
    corpus = docs + [question]

    #TF-IDF 向量: 用來從一段文字/一個語料庫中，給越重要的字詞/文檔，越高的加權分數。
    #字詞的重要性隨著 在文本出現的頻率越高則越高；在不同文本檔案間出現的次數越高則反而降低
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        #char_wb 以字元為單位切片，但會考慮單字邊界（word boundary），這對中英混合、中文、拼寫差異都比較有彈性
        ngram_range=(2, 4),
        #要取 2 到 4 個字元的片段，可以抓到局部字串相似性
        min_df=1
        #表示只要某個字元片段出現過 1 次，就保留
    )
    matrix = vectorizer.fit_transform(corpus)
    #用剛才TfidfVectorizer的定義遍歷資料，一邊把文字轉成 tf-idf 矩陣，通常用在第一次處理訓練資料 / 全部語料 的時候

    case_vectors = matrix[:-1] #把case、question分開
    question_vector = matrix[-1]

    #計算 問題向量 和 每一筆案例向量 的 cosine similarity（餘弦相似度）也就是兩個向量是不是在重要特徵上指向差不多的方向
    #cosine分數通常介於 0 ~ 1 之間，越大表示越相似
    # .flatten()把結果攤平成一維陣列，方便後面直接用索引取值。
    similarities = cosine_similarity(question_vector, case_vectors).flatten()

    rows = []

    #df.iterrows()會取出index + row，但是(_, row)代表只保留row
    #df的index做過篩選/刪除/排序/合併，pandas 通常會保留原本的 index，不會自動重編
    # enumerate 幫每列加上一個從 0 開始的編號 i，這個 i 可以對應到前面 similarities[i]
    for i, (_, row) in enumerate(df.iterrows()):
        sim = float(similarities[i])
        keyword_score = keyword_overlap_score(question, row["case_document"])
        final_score = (sim * 0.7) + (keyword_score * 0.3)

        rows.append({
            "Complaint ID": row.get("Complaint ID", ""),
            "Issue": row.get("Issue", ""),
            "Sub-issue": row.get("Sub-issue", ""),
            "Solution in Chinese": row.get("Solution in Chinese", ""),
            "similarity_score": sim,
            "keyword_score": keyword_score,
            "final_score": final_score
        })

    result_df = pd.DataFrame(rows).sort_values("final_score", ascending=False).reset_index(drop=True)

    # 把相似解法去重
    unique_solutions = [] 

    for _, row in result_df.iterrows():
        current_solution = row["Solution in Chinese"]

        is_similar = False
        for saved_row in unique_solutions: #只有第一圈是空的，之後會逐漸壯大
            sim_score = solution_similarity(current_solution, saved_row["Solution in Chinese"])
            if sim_score > 0.75:
                is_similar = True
                break

        if not is_similar:
            unique_solutions.append(row)

        if len(unique_solutions) >= top_k:
            break

    return pd.DataFrame(unique_solutions).reset_index(drop=True)

In [47]:
question = "消費者反映付款後沒有收到正確結果，該怎麼排查？"

In [49]:
top_results = rank_cases(question, df, top_k=3)

for i, row in top_results.iterrows():
    solution = str(row["Solution in Chinese"]).replace("建議處理方案（專業版）：", "").strip()

    print("=" * 80)
    print(f"建議方案 {i+1}")
    print(solution)
    print()

建議方案 1
1. 先以 ACH / direct deposit trace 流程追查款項，確認付款方檔案是否成功送達、銀行是否收到入帳訊息，以及是否因例外處理而落入未分配帳。
2. 由後台作業單位比對付款 ID、trace number、批次檔與核心系統入帳紀錄，必要時直接與付款機關/雇主對接。
3. 若確認款項已送達銀行但未正確入帳，應立即將 該筆款項 補登至客戶帳戶，並退還透支費、退件費及其他連帶費用。
4. 若涉及政府補助、薪資或受保護給付，應提高優先級處理並保留合規紀錄，避免不當延誤。
5. 以正式書面回覆客戶：說明資金路徑、延遲原因、補帳結果與後續防呆措施。

建議方案 2
1. 由支付作業單位核對每筆付款授權、取消回應碼、清算結果與帳務過帳紀錄，確認是否存在重複扣款或取消失敗。
2. 若系統已顯示交易失敗、取消或退回，應即刻沖回多扣資金，不得讓失敗交易持續佔用客戶餘額。
3. 檢視是否因此造成退票、租金/帳單延誤或其他連帶損失，並一併辦理費用沖銷或補償。
4. 如涉及第三方支付或外部商戶，應由銀行主動對接清算資訊，而非要求客戶自行追查。
5. 以書面通知客戶最終調整後之帳務明細、沖回金額與完成日。

建議方案 3
1. 由存款作業單位比對兩家銀行的票據影像、重複存入標記、退票/沖正紀錄及實際清算結果，確認是否出現雙重扣回或重複沖銷。
2. 明確區分『客戶誤重複提交』與『銀行系統重複扣帳』之責任，避免在未釐清前直接以負餘額方式處理。
3. 若確認客戶未取得雙重利益，應立即恢復正確餘額、返還 $1100.00 及任何因錯誤扣帳產生之費用。
4. 如工資或薪資遭自動抵銷負餘額，應一併返還被占用資金。
5. 提供書面調查說明，載明票據流向、沖正原因、修正後帳務與完成日期。

